# Robot Arm Training (MuJoCo)

This notebook trains a DDPG agent to control a 3-joint robot arm to reach random targets using MuJoCo physics.

In [ ]:
# 1. Install Dependencies
!pip install mujoco torch numpy matplotlib

In [ ]:
# 2. Create MuJoCo Model File
xml_content = """
<mujoco model="robot_arm">
  <compiler angle="radian" meshdir="."/>
  <option timestep="0.00416" gravity="0 0 -9.81"/>
  <visual>
    <headlight diffuse="0.6 0.6 0.6" ambient="0.1 0.1 0.1" specular="0 0 0"/>
    <rgba haze="0.15 0.25 0.35 1"/>
    <global azimuth="120" elevation="-20"/>
  </visual>
  <asset>
    <texture type="skybox" builtin="gradient" rgb1="0.3 0.5 0.7" rgb2="0 0 0" width="512" height="3072"/>
    <texture type="2d" name="groundplane" builtin="checker" rgb1="0.2 0.3 0.4" rgb2="0.1 0.2 0.3" width="100" height="100"/>
    <material name="groundplane" texture="groundplane" texrepeat="10 10" reflectance="0.2"/>
    <material name="dark_gray" rgba="0.3 0.3 0.3 1"/>
    <material name="blue" rgba="0.2 0.4 0.8 1"/>
    <material name="teal" rgba="0.1 0.6 0.6 1"/>
    <material name="orange" rgba="0.9 0.5 0.1 1"/>
    <material name="green" rgba="0.1 0.9 0.2 1"/>
    <material name="red" rgba="1 0.2 0.2 0.8"/>
  </asset>
  <worldbody>
    <light pos="0 0 1.5" dir="0 0 -1" directional="true"/>
    <geom name="floor" size="0 0 0.05" type="plane" material="groundplane"/>
    <body name="base_link" pos="0 0 0.025">
      <geom type="cylinder" size="0.1 0.025" material="dark_gray"/>
      <body name="link_1" pos="0 0 0.05"> 
        <joint name="joint_1" type="hinge" axis="0 0 1" range="-3.14159 3.14159" damping="0.5" frictionloss="0.1"/>
        <geom type="cylinder" size="0.05 0.2" pos="0 0 0.2" material="blue"/>
        <body name="link_2" pos="0 0 0.4">
          <joint name="joint_2" type="hinge" axis="0 1 0" range="-3.14159 3.14159" damping="0.5" frictionloss="0.1"/>
          <geom type="cylinder" size="0.04 0.2" pos="0 0 0.2" material="teal"/>
          <body name="link_3" pos="0 0 0.4">
            <joint name="joint_3" type="hinge" axis="0 1 0" range="-3.14159 3.14159" damping="0.5" frictionloss="0.1"/>
            <geom type="cylinder" size="0.03 0.15" pos="0 0 0.15" material="orange"/>
            <body name="ee_link" pos="0 0 0.3">
               <geom type="sphere" size="0.03" material="green"/>
               <site name="ee_site" pos="0 0 0" size="0.01" rgba="1 0 0 0"/>
            </body>
          </body>
        </body>
      </body>
    </body>
    <body name="target" mocap="true" pos="0.5 0 0.5">
      <geom type="sphere" size="0.05" material="red" contype="0" conaffinity="0"/>
    </body>
  </worldbody>
  <actuator>
    <motor name="actuator_1" joint="joint_1" gear="1" ctrllimited="true" ctrlrange="-50 50"/>
    <motor name="actuator_2" joint="joint_2" gear="1" ctrllimited="true" ctrlrange="-50 50"/>
    <motor name="actuator_3" joint="joint_3" gear="1" ctrllimited="true" ctrlrange="-50 50"/>
  </actuator>
</mujoco>
"""

with open("robot_arm.xml", "w") as f:
    f.write(xml_content)

In [ ]:
# 3. Import Libraries
import os
import math
import copy
import numpy as np
import mujoco
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt

# Set device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

In [ ]:
# 4. Define Environment
class RobotArmEnv:
    def __init__(self, max_steps=500):
        self.max_steps = max_steps
        self.step_count = 0

        # Load MuJoCo model
        self.model = mujoco.MjModel.from_xml_path("robot_arm.xml")
        self.data = mujoco.MjData(self.model)

        # IDs
        self.ee_site_id = mujoco.mj_name2id(self.model, mujoco.mjtObj.mjOBJ_SITE, "ee_site")
        self.target_body_id = mujoco.mj_name2id(self.model, mujoco.mjtObj.mjOBJ_BODY, "target")
        
        self.joint_indices = [0, 1, 2] 
        self.num_joints = 3
        self.max_torque = 50.0
        self.state_dim = 12
        self.action_dim = 3
        self.target_pos = np.zeros(3)

    def reset(self):
        mujoco.mj_resetData(self.model, self.data)
        for i in range(self.num_joints):
            self.data.qpos[i] = np.random.uniform(-0.3, 0.3)
            self.data.qvel[i] = 0.0
        self._spawn_target()
        mujoco.mj_forward(self.model, self.data)
        self.step_count = 0
        return self._get_obs()

    def step(self, action):
        action = np.clip(action, -1.0, 1.0)
        torques = action * self.max_torque
        self.data.ctrl[:] = torques
        mujoco.mj_step(self.model, self.data)
        self.step_count += 1
        obs = self._get_obs()
        dist = self._get_distance()
        reward = -dist
        done = self.step_count >= self.max_steps
        info = {"distance": dist}
        return obs, reward, done, info

    def _get_obs(self):
        angles = self.data.qpos[:3].copy()
        velocities = self.data.qvel[:3].copy()
        ee_pos = self.data.site_xpos[self.ee_site_id].copy()
        vec_to_target = self.target_pos - ee_pos
        # Normalize
        obs = np.concatenate([angles / np.pi, velocities * 0.1, self.target_pos, vec_to_target])
        return obs.astype(np.float32)

    def _get_distance(self):
        ee_pos = self.data.site_xpos[self.ee_site_id]
        return float(np.linalg.norm(ee_pos - self.target_pos))

    def _spawn_target(self):
        r = np.random.uniform(0.3, 0.8)
        theta = np.random.uniform(0, 2 * math.pi)
        z = np.random.uniform(0.2, 0.9)
        x = r * math.cos(theta)
        y = r * math.sin(theta)
        self.target_pos = np.array([x, y, z], dtype=np.float32)
        self.data.mocap_pos[0] = self.target_pos

In [ ]:
# 5. Define Agent & Replay Buffer

class Actor(nn.Module):
    def __init__(self, state_dim, action_dim):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(state_dim, 256), nn.ReLU(),
            nn.Linear(256, 256), nn.ReLU(),
            nn.Linear(256, 128), nn.ReLU(),
            nn.Linear(128, action_dim), nn.Tanh()
        )

    def forward(self, state):
        return self.net(state)

class Critic(nn.Module):
    def __init__(self, state_dim, action_dim):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(state_dim + action_dim, 256), nn.ReLU(),
            nn.Linear(256, 256), nn.ReLU(),
            nn.Linear(256, 128), nn.ReLU(),
            nn.Linear(128, 1)
        )

    def forward(self, state, action):
        return self.net(torch.cat([state, action], dim=-1))

class OUNoise:
    def __init__(self, size, mu=0.0, theta=0.15, sigma=0.2):
        self.mu = mu * np.ones(size)
        self.theta = theta
        self.sigma = sigma
        self.state = np.copy(self.mu)

    def reset(self):
        self.state = np.copy(self.mu)

    def sample(self):
        dx = self.theta * (self.mu - self.state) + self.sigma * np.random.randn(len(self.mu))
        self.state += dx
        return self.state.astype(np.float32)

class ReplayBuffer:
    def __init__(self, state_dim, action_dim, capacity=1_000_000):
        self.capacity = capacity
        self.size = 0
        self.ptr = 0
        self.states = np.zeros((capacity, state_dim), dtype=np.float32)
        self.actions = np.zeros((capacity, action_dim), dtype=np.float32)
        self.rewards = np.zeros((capacity, 1), dtype=np.float32)
        self.next_states = np.zeros((capacity, state_dim), dtype=np.float32)
        self.dones = np.zeros((capacity, 1), dtype=np.float32)

    def add(self, state, action, reward, next_state, done):
        self.states[self.ptr] = state
        self.actions[self.ptr] = action
        self.rewards[self.ptr] = reward
        self.next_states[self.ptr] = next_state
        self.dones[self.ptr] = float(done)
        self.ptr = (self.ptr + 1) % self.capacity
        self.size = min(self.size + 1, self.capacity)

    def sample(self, batch_size=128):
        indices = np.random.randint(0, self.size, size=batch_size)
        return (
            self.states[indices], self.actions[indices],
            self.rewards[indices], self.next_states[indices],
            self.dones[indices]
        )
    def __len__(self): return self.size

class DDPGAgent:
    def __init__(self, state_dim, action_dim, actor_lr=1e-4, critic_lr=1e-3, gamma=0.99, tau=0.005):
        self.gamma = gamma
        self.tau = tau
        self.actor = Actor(state_dim, action_dim).to(device)
        self.actor_target = copy.deepcopy(self.actor)
        self.actor_optimizer = torch.optim.Adam(self.actor.parameters(), lr=actor_lr)
        self.critic = Critic(state_dim, action_dim).to(device)
        self.critic_target = copy.deepcopy(self.critic)
        self.critic_optimizer = torch.optim.Adam(self.critic.parameters(), lr=critic_lr)
        self.noise = OUNoise(action_dim)

    def select_action(self, state, add_noise=True):
        state_t = torch.FloatTensor(state).unsqueeze(0).to(device)
        with torch.no_grad():
            action = self.actor(state_t).cpu().numpy()[0]
        if add_noise:
            action += self.noise.sample()
        return np.clip(action, -1.0, 1.0)

    def update(self, replay_buffer, batch_size=128):
        states, actions, rewards, next_states, dones = replay_buffer.sample(batch_size)
        states = torch.FloatTensor(states).to(device)
        actions = torch.FloatTensor(actions).to(device)
        rewards = torch.FloatTensor(rewards).to(device)
        next_states = torch.FloatTensor(next_states).to(device)
        dones = torch.FloatTensor(dones).to(device)

        with torch.no_grad():
            next_actions = self.actor_target(next_states)
            target_q = self.critic_target(next_states, next_actions)
            y = rewards + (1 - dones) * self.gamma * target_q

        current_q = self.critic(states, actions)
        critic_loss = F.mse_loss(current_q, y)
        self.critic_optimizer.zero_grad()
        critic_loss.backward()
        self.critic_optimizer.step()

        actor_loss = -self.critic(states, self.actor(states)).mean()
        self.actor_optimizer.zero_grad()
        actor_loss.backward()
        self.actor_optimizer.step()

        self._soft_update(self.actor, self.actor_target)
        self._soft_update(self.critic, self.critic_target)

    def _soft_update(self, source, target):
        for sp, tp in zip(source.parameters(), target.parameters()):
            tp.data.copy_(self.tau * sp.data + (1.0 - self.tau) * tp.data)

    def save(self, directory="models"):
        os.makedirs(directory, exist_ok=True)
        torch.save(self.actor.state_dict(), os.path.join(directory, "actor.pth"))
        torch.save(self.critic.state_dict(), os.path.join(directory, "critic.pth"))

In [ ]:
# 6. Train Loop
env = RobotArmEnv()
agent = DDPGAgent(state_dim=env.state_dim, action_dim=env.action_dim)
buffer = ReplayBuffer(state_dim=env.state_dim, action_dim=env.action_dim)

num_episodes = 1000
batch_size = 128
warmup_steps = 1000

print("Starting training...")
total_steps = 0
rewards_history = []

for episode in range(1, num_episodes + 1):
    state = env.reset()
    agent.noise.reset()
    episode_reward = 0

    for step in range(env.max_steps):
        if total_steps < warmup_steps:
            action = np.random.uniform(-1, 1, size=env.action_dim).astype(np.float32)
        else:
            action = agent.select_action(state, add_noise=True)

        next_state, reward, done, info = env.step(action)
        buffer.add(state, action, reward, next_state, done)
        episode_reward += reward
        state = next_state
        total_steps += 1
        
        # Train every step to speed up learning
        if len(buffer) >= batch_size and total_steps >= warmup_steps:
            agent.update(buffer, batch_size)
        
        if done: break

    rewards_history.append(episode_reward)
    if episode % 10 == 0:
        print(f"Ep {episode} | Reward: {episode_reward:.1f} | Avg (last 10): {np.mean(rewards_history[-10:]):.1f}")

agent.save("models_colab")
print("Training finished.")

In [ ]:
# 7. Download Models
from google.colab import files
!zip -r models_colab.zip models_colab
files.download('models_colab.zip')